# DICOM QC и геометрия тома

**Статус:** активный производитель кандидатного DICOM-манифеста.
Результат становится допустимым входом следующих этапов только после
ручного контроля и статуса accepted.

Ноутбук решает одну задачу: инвентаризирует выбранную КТ-серию и проверяет
геометрию тома в системе координат пациента DICOM LPS. Он не загружает
пиксельные данные, не сегментирует лёгкие, не вычисляет HU, долю воздуха,
удельное сопротивление или положение электродов.

Исторический объединённый расчёт до разделения доступен в Git в коммите
fff4d3a. Указатель происхождения находится в
archive/legacy/20.90_Скетч_пороговой_КТ_сегментации.ipynb. Его части
распределены так: DICOM-QC — здесь, регистрация ручных масок Inobitec —
в 20.02, HU и доля воздуха — в 20.03, электрические сценарии — в 20.04.


## Входы, происхождение и ограничения

Непосредственно измеренные входы — заголовки DICOM каждого среза:
идентификаторы серии и изображения, положение и ориентация в системе
координат пациента, размер матрицы, шаг пикселя и параметры преобразования
в HU. Пиксельные данные на этом этапе не читаются.

Из внешней локальной конфигурации поступают обезличенный идентификатор
испытуемого, путь к каталогу, точный селектор серии, состояние дыхания и
статус контрастирования вместе с источниками этих сведений. Эти поля не
считаются автоматически подтверждёнными DICOM-заголовком.

Рабочие допуски однородности ориентации и шага задаются в конфигурации.
Это инженерные пороги QC, а не универсальные физиологические или
метрологические константы. Их значения и влияние на решение о пригодности
тома следует обосновать отдельно.

Сырые КТ, реальные пути, DICOM UID и производные субъектно-связанные данные
остаются вне Git. Манифест не сохраняет имя пациента, дату исследования,
исходные UID и имена файлов: для прослеживаемости используются SHA-256.

Синтетический тест выполнен в Python 3.13.6 с pydicom 3.0.2. Другие версии
среды пока не проверены; фактические версии сохраняются в каждом манифесте.


## Проверяемая геометрия

Направления строк и столбцов берутся из ImageOrientationPatient. Нормаль к
срезу рассчитывается как их векторное произведение. Для положения среза
p_i вводится координата вдоль нормали

$$
s_i = n \mathbin{\cdot} p_i.
$$

После сортировки по s_i проверяются повторяющиеся позиции, однородность
межсрезового шага, постоянство ориентации, PixelSpacing, Rows и Columns.
Такой расчёт не предполагает, что меньший индекс массива означает правую
сторону пациента или нижнюю часть лёгкого.

Отдельно фиксируются наличие метаданных о контрасте, ядро реконструкции и
параметры RescaleSlope/RescaleIntercept. Отсутствие тега контрастирования
само по себе не доказывает, что исследование выполнено без контраста.


## Внешняя конфигурация

Создайте локальную копию config/ct_paths.example.json, заполните её без
добавления в Git и задайте абсолютный путь переменной
KALMYKOV_CT_CONFIG. Селектор серии должен однозначно выбрать один
SeriesInstanceUID; совпадение только по названию допускается лишь при
отсутствии неоднозначности.


In [ ]:
import hashlib
import json
import math
import os
import platform
import statistics
from datetime import datetime, timezone
from importlib.metadata import version as package_version
from pathlib import Path

import pydicom
from pydicom.errors import InvalidDicomError

ALGORITHM_VERSION = "dicom_qc_geometry_v1"
CONFIG_PATH = Path(os.environ["KALMYKOV_CT_CONFIG"]).expanduser().resolve()
CONFIG_BYTES = CONFIG_PATH.read_bytes()
CONFIG = json.loads(CONFIG_BYTES.decode("utf-8"))
DERIVED_ROOT = Path(CONFIG["derived_root"]).expanduser().resolve()
QC_SETTINGS = CONFIG["dicom_qc"]
SUBJECT_SPECS = CONFIG["subjects"]

subject_ids = [item["subject_id"] for item in SUBJECT_SPECS]
if len(subject_ids) != len(set(subject_ids)):
    raise ValueError("subject_id должны быть уникальными")
if not SUBJECT_SPECS:
    raise ValueError("Конфигурация не содержит испытуемых")

RUNTIME = {
    "python": platform.python_version(),
    "pydicom": package_version("pydicom"),
}


In [ ]:
REQUIRED_TAGS = [
    "Modality",
    "SOPClassUID",
    "SOPInstanceUID",
    "SeriesInstanceUID",
    "SeriesDescription",
    "SeriesNumber",
    "ImagePositionPatient",
    "ImageOrientationPatient",
    "PixelSpacing",
    "Rows",
    "Columns",
    "SliceThickness",
    "SpacingBetweenSlices",
    "RescaleSlope",
    "RescaleIntercept",
    "ConvolutionKernel",
    "ContrastBolusAgent",
    "ContrastBolusVolume",
    "ImageType",
]


def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def hashed_text(value):
    if value is None:
        return None
    return sha256_bytes(str(value).encode("utf-8"))


def float_vector(value, length, field):
    if value is None or len(value) != length:
        raise ValueError(f"{field}: ожидается {length} чисел")
    return [float(item) for item in value]


def dot(left, right):
    return sum(a * b for a, b in zip(left, right))


def cross(left, right):
    return [
        left[1] * right[2] - left[2] * right[1],
        left[2] * right[0] - left[0] * right[2],
        left[0] * right[1] - left[1] * right[0],
    ]


def norm(vector):
    return math.sqrt(dot(vector, vector))


def normalize(vector):
    length = norm(vector)
    if length == 0:
        raise ValueError("Нулевой направляющий вектор DICOM")
    return [item / length for item in vector]


def max_abs_difference(left, right):
    return max(abs(a - b) for a, b in zip(left, right))


def dicom_files(folder, recursive):
    iterator = folder.rglob("*") if recursive else folder.iterdir()
    return sorted(path for path in iterator if path.is_file())


def read_series(folder, recursive=True):
    groups = {}
    rejected_files = 0
    for path in dicom_files(folder, recursive):
        try:
            dataset = pydicom.dcmread(
                path,
                stop_before_pixels=True,
                specific_tags=REQUIRED_TAGS,
            )
        except (InvalidDicomError, OSError, ValueError):
            rejected_files += 1
            continue
        if str(getattr(dataset, "Modality", "")) != "CT":
            continue
        series_uid = str(getattr(dataset, "SeriesInstanceUID", "")).strip()
        if not series_uid:
            rejected_files += 1
            continue
        groups.setdefault(series_uid, []).append((path, dataset))
    return groups, rejected_files


def selector_matches(dataset, selector):
    checks = []
    if selector.get("series_instance_uid"):
        checks.append(
            str(dataset.SeriesInstanceUID) == str(selector["series_instance_uid"])
        )
    if selector.get("series_number") is not None:
        checks.append(
            str(getattr(dataset, "SeriesNumber", ""))
            == str(selector["series_number"])
        )
    if selector.get("series_description"):
        checks.append(
            str(getattr(dataset, "SeriesDescription", "")).strip()
            == str(selector["series_description"]).strip()
        )
    if not checks:
        raise ValueError("selected_series не содержит ни одного селектора")
    return all(checks)


def select_one_series(groups, selector):
    matches = [
        (series_uid, records)
        for series_uid, records in groups.items()
        if selector_matches(records[0][1], selector)
    ]
    if len(matches) != 1:
        candidates = [
            {
                "series_uid_sha256": hashed_text(uid),
                "series_number": str(getattr(records[0][1], "SeriesNumber", "")),
                "series_description_sha256": hashed_text(
                    str(getattr(records[0][1], "SeriesDescription", "")).strip()
                ),
                "slice_count": len(records),
            }
            for uid, records in groups.items()
        ]
        raise RuntimeError(
            "Селектор должен выбрать ровно одну серию; "
            f"найдено {len(matches)}. Обезличенные кандидаты: {candidates}"
        )
    return matches[0]


In [ ]:
def geometry_manifest(subject, series_uid, records, rejected_files):
            errors = []
            warnings = []
            position_tolerance = float(QC_SETTINGS["duplicate_position_tolerance_mm"])
            orientation_tolerance = float(QC_SETTINGS["orientation_tolerance"])
            pixel_spacing_tolerance = float(QC_SETTINGS["pixel_spacing_tolerance_mm"])
            slice_spacing_tolerance = float(QC_SETTINGS["slice_spacing_tolerance_mm"])

            required = [
                "SOPInstanceUID",
                "ImagePositionPatient",
                "ImageOrientationPatient",
                "PixelSpacing",
                "Rows",
                "Columns",
            ]
            missing = {
                field: sum(not hasattr(dataset, field) for _, dataset in records)
                for field in required
            }
            missing = {field: count for field, count in missing.items() if count}
            if missing:
                errors.append({"code": "missing_required_tags", "counts": missing})

            geometry = None
            if not errors:
                orientations = [
                    float_vector(dataset.ImageOrientationPatient, 6, "ImageOrientationPatient")
                    for _, dataset in records
                ]
                reference_orientation = orientations[0]
                row = normalize(reference_orientation[:3])
                column = normalize(reference_orientation[3:])
                normal = normalize(cross(row, column))
                orientation_deviation = max(
                    max_abs_difference(item, reference_orientation)
                    for item in orientations
                )
                if orientation_deviation > orientation_tolerance:
                    errors.append(
                        {
                            "code": "nonuniform_orientation",
                            "max_abs_deviation": orientation_deviation,
                        }
                    )
                if abs(dot(row, column)) > orientation_tolerance:
                    errors.append(
                        {
                            "code": "nonorthogonal_orientation",
                            "absolute_dot_product": abs(dot(row, column)),
                        }
                    )

                positions = [
                    float_vector(dataset.ImagePositionPatient, 3, "ImagePositionPatient")
                    for _, dataset in records
                ]
                coordinates = sorted(dot(normal, item) for item in positions)
                coordinate_steps = [
                    coordinates[index + 1] - coordinates[index]
                    for index in range(len(coordinates) - 1)
                ]
                duplicate_positions = sum(
                    abs(step) <= position_tolerance for step in coordinate_steps
                )
                if duplicate_positions:
                    errors.append(
                        {
                            "code": "duplicate_slice_positions",
                            "count": duplicate_positions,
                        }
                    )

                positive_steps = [
                    step for step in coordinate_steps if step > position_tolerance
                ]
                median_slice_spacing = (
                    statistics.median(positive_steps) if positive_steps else None
                )
                spacing_deviation = (
                    max(abs(step - median_slice_spacing) for step in positive_steps)
                    if positive_steps and median_slice_spacing is not None
                    else None
                )
                if spacing_deviation is None:
                    errors.append({"code": "insufficient_distinct_slice_positions"})
                elif spacing_deviation > slice_spacing_tolerance:
                    errors.append(
                        {
                            "code": "nonuniform_slice_spacing",
                            "max_deviation_mm": spacing_deviation,
                        }
                    )

                pixel_spacings = [
                    float_vector(dataset.PixelSpacing, 2, "PixelSpacing")
                    for _, dataset in records
                ]
                reference_pixel_spacing = pixel_spacings[0]
                pixel_spacing_deviation = max(
                    max_abs_difference(item, reference_pixel_spacing)
                    for item in pixel_spacings
                )
                if pixel_spacing_deviation > pixel_spacing_tolerance:
                    errors.append(
                        {
                            "code": "nonuniform_pixel_spacing",
                            "max_deviation_mm": pixel_spacing_deviation,
                        }
                    )

                dimensions = {
                    (int(dataset.Rows), int(dataset.Columns))
                    for _, dataset in records
                }
                if len(dimensions) != 1:
                    errors.append(
                        {
                            "code": "nonuniform_matrix_size",
                            "sizes": sorted([list(item) for item in dimensions]),
                        }
                    )

                sop_uids = [
                    str(dataset.SOPInstanceUID).strip() for _, dataset in records
                ]
                duplicate_sop_uids = len(sop_uids) - len(set(sop_uids))
                if duplicate_sop_uids:
                    errors.append(
                        {
                            "code": "duplicate_sop_instance_uid",
                            "count": duplicate_sop_uids,
                        }
                    )

                geometry = {
                    "coordinate_system": "DICOM patient LPS",
                    "slice_count": len(records),
                    "matrix_rows_columns": list(next(iter(dimensions)))
                    if len(dimensions) == 1
                    else None,
                    "pixel_spacing_mm": reference_pixel_spacing,
                    "pixel_spacing_max_deviation_mm": pixel_spacing_deviation,
                    "orientation_row_lps": row,
                    "orientation_column_lps": column,
                    "slice_normal_lps": normal,
                    "orientation_max_abs_deviation": orientation_deviation,
                    "position_range_mm_along_normal": [
                        min(coordinates),
                        max(coordinates),
                    ],
                    "median_slice_spacing_mm": median_slice_spacing,
                    "slice_spacing_max_deviation_mm": spacing_deviation,
                }

            datasets = [dataset for _, dataset in records]
            contrast_metadata_present = any(
                bool(str(getattr(dataset, field, "")).strip())
                for dataset in datasets
                for field in ("ContrastBolusAgent", "ContrastBolusVolume")
            )
            if contrast_metadata_present:
                warnings.append({"code": "contrast_metadata_present"})
            if subject.get("contrast_status") == "unknown":
                warnings.append({"code": "contrast_status_requires_primary_source"})
            if subject.get("breathing_state") == "unknown":
                warnings.append({"code": "breathing_state_requires_primary_source"})

            file_hashes = sorted(sha256_file(path) for path, _ in records)
            source_set_sha256 = sha256_bytes(
                ("\n".join(file_hashes) + "\n").encode("ascii")
            )
            selector = subject["selected_series"]
            first = datasets[0]
            return {
                "schema_version": 1,
                "algorithm_version": ALGORITHM_VERSION,
                "created_at": datetime.now(timezone.utc).isoformat(),
                "subject_id": subject["subject_id"],
                "source": {
                    "config_sha256": sha256_bytes(CONFIG_BYTES),
                    "series_instance_uid_sha256": hashed_text(series_uid),
                    "series_description_sha256": hashed_text(
                        str(getattr(first, "SeriesDescription", "")).strip()
                    ),
                    "series_number": str(getattr(first, "SeriesNumber", "")),
                    "selector": {
                        "series_instance_uid_sha256": hashed_text(
                            selector.get("series_instance_uid")
                        ),
                        "series_description_sha256": hashed_text(
                            selector.get("series_description")
                        ),
                        "series_number": selector.get("series_number"),
                    },
                    "source_file_sha256": file_hashes,
                    "source_set_sha256": source_set_sha256,
                    "rejected_non_dicom_or_invalid_files": rejected_files,
                },
                "acquisition_context": {
                    "breathing_state": subject.get("breathing_state", "unknown"),
                    "breathing_state_source": subject.get(
                        "breathing_state_source"
                    ),
                    "contrast_status": subject.get("contrast_status", "unknown"),
                    "contrast_status_source": subject.get(
                        "contrast_status_source"
                    ),
                    "contrast_metadata_present": contrast_metadata_present,
                    "convolution_kernel_values": sorted(
                        {
                            str(getattr(dataset, "ConvolutionKernel", "")).strip()
                            for dataset in datasets
                            if str(getattr(dataset, "ConvolutionKernel", "")).strip()
                        }
                    ),
                    "rescale_slope_values": sorted(
                        {
                            float(getattr(dataset, "RescaleSlope", 1.0))
                            for dataset in datasets
                        }
                    ),
                    "rescale_intercept_values": sorted(
                        {
                            float(getattr(dataset, "RescaleIntercept", 0.0))
                            for dataset in datasets
                        }
                    ),
                },
                "geometry": geometry,
                "runtime": RUNTIME,
                "qc": {
                    "status": "pending_manual_review",
                    "automatic_errors": errors,
                    "automatic_warnings": warnings,
                    "reviewer": None,
                    "reviewed_at": None,
                    "notes": None,
                    "history": [],
                },
            }


## Формирование кандидатного манифеста

Для каждого испытуемого код выбирает ровно одну серию и создаёт внешний
файл derived_root/ct/dicom_qc/<subject_id>.json. Существующий файл не
перезаписывается. Если манифест относится к другому набору файлов,
конфигурации или версии алгоритма, расчёт останавливается и требует
отдельного архивирования и повторного контроля.


In [ ]:
MANIFESTS = {}
output_root = DERIVED_ROOT / "ct" / "dicom_qc"
output_root.mkdir(parents=True, exist_ok=True)

for subject in SUBJECT_SPECS:
    folder = Path(subject["dicom_dir"]).expanduser().resolve()
    if not folder.is_dir():
        raise FileNotFoundError(
            f"Каталог DICOM для {subject['subject_id']} не найден"
        )
    groups, rejected_files = read_series(
        folder,
        recursive=bool(subject.get("recursive", True)),
    )
    series_uid, records = select_one_series(
        groups,
        subject["selected_series"],
    )
    candidate = geometry_manifest(
        subject,
        series_uid,
        records,
        rejected_files,
    )
    output_path = output_root / f"{subject['subject_id']}.json"
    if output_path.exists():
        existing = json.loads(output_path.read_text(encoding="utf-8"))
        same_input = (
            existing.get("source", {}).get("source_set_sha256")
            == candidate["source"]["source_set_sha256"]
        )
        same_algorithm = (
            existing.get("algorithm_version") == ALGORITHM_VERSION
        )
        same_config = (
            existing.get("source", {}).get("config_sha256")
            == candidate["source"]["config_sha256"]
        )
        if not same_input or not same_algorithm or not same_config:
            raise RuntimeError(
                f"Существующий манифест {subject['subject_id']} относится "
                "к другому входу, конфигурации или версии алгоритма; "
                "сначала архивируйте его"
            )
        MANIFESTS[subject["subject_id"]] = existing
        print(
            "Сохранён существующий манифест:",
            subject["subject_id"],
            existing["qc"]["status"],
        )
        continue

    output_path.write_text(
        json.dumps(candidate, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    MANIFESTS[subject["subject_id"]] = candidate
    print(
        "Создан кандидатный манифест:",
        subject["subject_id"],
        "ошибок:",
        len(candidate["qc"]["automatic_errors"]),
        "предупреждений:",
        len(candidate["qc"]["automatic_warnings"]),
    )


## Ручной контроль и принятие

Перед принятием необходимо сверить выбранную серию с первичным протоколом
или отчётом исследования, проверить дыхательное состояние и наличие
контраста, а также просмотреть автоматические ошибки и предупреждения.
Статус accepted требует отсутствия автоматических ошибок, имени
проверяющего, явного подтверждения выбранной серии, состояния дыхания и
статуса контрастирования.

Контроль этого этапа подтверждает только целостность входной серии и её
геометрического описания. Он не подтверждает качество сегментации, ROI,
HU-модель или электрические параметры тканей.


In [ ]:
REVIEW_DECISIONS = {
    # "exp02_nik": {
    #     "status": "accepted",  # accepted или rejected
    #     "reviewer": "<reviewer>",
    #     "confirmed_series_selection": True,
    #     "confirmed_breathing_state": "inhale_breath_hold",
    #     "confirmed_contrast_status": "noncontrast",
    #     "notes": "<основание решения>",
    # },
}

for subject_id, decision in REVIEW_DECISIONS.items():
    if subject_id not in MANIFESTS:
        raise KeyError(f"Неизвестный subject_id: {subject_id}")
    manifest = MANIFESTS[subject_id]
    if manifest["qc"]["status"] == "accepted":
        print("Уже принят, без перезаписи:", subject_id)
        continue

    status = decision.get("status")
    reviewer = str(decision.get("reviewer", "")).strip()
    if status not in {"accepted", "rejected"} or not reviewer:
        raise ValueError(
            "Нужны статус accepted/rejected и имя проверяющего"
        )
    if status == "accepted":
        if manifest["qc"]["automatic_errors"]:
            raise ValueError(
                f"{subject_id}: нельзя принять манифест с ошибками QC"
            )
        if decision.get("confirmed_series_selection") is not True:
            raise ValueError("Требуется подтвердить выбор серии")
        if not decision.get("confirmed_breathing_state"):
            raise ValueError("Требуется подтвердить состояние дыхания")
        if decision.get("confirmed_contrast_status") not in {
            "noncontrast",
            "contrast",
        }:
            raise ValueError(
                "confirmed_contrast_status: noncontrast или contrast"
            )

    previous_qc = manifest["qc"]
    history = list(previous_qc.get("history", []))
    history.append(
        {
            "status": previous_qc.get("status"),
            "reviewer": previous_qc.get("reviewer"),
            "reviewed_at": previous_qc.get("reviewed_at"),
            "notes": previous_qc.get("notes"),
        }
    )
    manifest["acquisition_context"]["breathing_state"] = decision.get(
        "confirmed_breathing_state",
        manifest["acquisition_context"]["breathing_state"],
    )
    manifest["acquisition_context"]["contrast_status"] = decision.get(
        "confirmed_contrast_status",
        manifest["acquisition_context"]["contrast_status"],
    )
    manifest["qc"] = {
        "status": status,
        "automatic_errors": previous_qc["automatic_errors"],
        "automatic_warnings": previous_qc["automatic_warnings"],
        "reviewer": reviewer,
        "reviewed_at": datetime.now(timezone.utc).isoformat(),
        "notes": decision.get("notes"),
        "history": history,
    }
    output_path = output_root / f"{subject_id}.json"
    output_path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print("Решение сохранено:", subject_id, status)


## Выход и зависимые этапы

Канонический выход — внешний манифест
derived_root/ct/dicom_qc/<subject_id>.json. Следующие этапы имеют право
использовать только манифест той же версии алгоритма со статусом accepted
и совпадающим SHA-256 исходного набора DICOM.

- 20.02 связывает принятую геометрию тома с ручными масками Inobitec и
  геометрией электродов.
- 20.03 читает принятую геометрию и независимо заданный ROI для анализа HU.
- 20.04 получает только модельно-условные сценарии электрических свойств из
  результатов 20.03.

params/ct.json не является выходом этого ноутбука. Существующие записи в
нём относятся к историческому смешанному расчёту и до отдельной миграции
не считаются проверенным продуктом серии 20.
